In [2]:
import numpy as np
import pandas as pd
import joblib
import os
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

BASE = r'C:\Users\Anushka Thite\Desktop\smart-loan-system'
PROCESSED_DIR = os.path.join(BASE, 'data', 'processed')

# Use SQLite backend (recommended in MLflow 3.x)
DB_PATH = os.path.join(BASE, 'mlflow.db').replace(chr(92), '/')
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
mlflow.set_experiment("loan_approval_models")

X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
y_test = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}\n")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              random_state=42, eval_metric='logloss', n_jobs=-1),
}

for name, model in models.items():
    print(f"{'=' * 50}")
    print(f"Training: {name}")
    print(f"{'=' * 50}")

    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)

        mlflow.log_param("model_name", name)
        mlflow.log_param("n_estimators", getattr(model, 'n_estimators', 'N/A'))
        mlflow.log_param("max_depth", getattr(model, 'max_depth', 'N/A'))
        mlflow.log_param("learning_rate", getattr(model, 'learning_rate', 'N/A'))

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("auc", auc)

        print(f"  Accuracy : {acc:.4f}")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall   : {rec:.4f}")
        print(f"  F1       : {f1:.4f}")
        print(f"  AUC      : {auc:.4f}")
        print()

print("\n✅ All models tracked in MLflow.")

2026/10/07 18:04:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/07 18:04:07 INFO mlflow.store.db.utils: Updating database tables
2026/10/07 18:04:11 INFO mlflow.tracking.fluent: Experiment with name 'loan_approval_models' does not exist. Creating a new experiment.


Loaded: X_train=(20570, 63), X_test=(4000, 63)
Tracking URI: sqlite:///C:/Users/Anushka Thite/Desktop/smart-loan-system/mlflow.db

Training: Logistic Regression
  Accuracy : 0.9487
  Precision: 0.9621
  Recall   : 0.9580
  F1       : 0.9600
  AUC      : 0.9897

Training: KNN
  Accuracy : 0.9100
  Precision: 0.9434
  Recall   : 0.9148
  F1       : 0.9289
  AUC      : 0.9628

Training: Random Forest
  Accuracy : 0.9457
  Precision: 0.9482
  Recall   : 0.9685
  F1       : 0.9582
  AUC      : 0.9899

Training: XGBoost
  Accuracy : 0.9527
  Precision: 0.9529
  Recall   : 0.9747
  F1       : 0.9637
  AUC      : 0.9919


✅ All models tracked in MLflow.
